# Evolutionary Factor Researcher — a **live** mini-run

The companion notebook `evolutionary_factor_researcher_walkthrough.ipynb` dissects
every part on a *synthetic* panel with the LLM calls switched **off**, so it stays
free and deterministic.  This notebook is the opposite: a **tiny end-to-end run of
the real machine** so you (and a supervisor) can watch it actually turn —

* **real LLM calls** — brainstorm → codegen → reflection → mutation, all against
  the live model (`gpt-4o-mini` by default),
* **a pre-settled book** — a Lasso-selected subset of the existing
  `yfinance_equity_sp100` factors is loaded as fixed context, and
* **real ML fitting** — every candidate is scored by the deterministic
  `research_eval` harness, which combines the fixed book's factor signals into one
  forward-return forecast and measures the candidate's **marginal out-of-sample
  IC** (LOCO), residual IC, CPCV robustness, deflation, and the hard gates,

on a small slice of **real S&P-100 daily data** (served offline from the local
`yfinance` parquet cache — no network needed for the data).

> **The one invariant of the whole design:** the LLM *mutates*, the deterministic
> harness *scores*.  No LLM output ever reaches the reward channel except as code
> to compile — so the model can never influence its own fitness.

This run makes a couple dozen cheap API calls and takes a few minutes.


## 0 · Setup — keys, data cache, run knobs


In [ ]:
import os, sys, json, time, textwrap, logging
from pathlib import Path

import pandas as pd, numpy as np

# --- locate the repo root (the dir that contains quant_fund_agent/) ---
here = Path.cwd()
ROOT = next((p for p in [here, *here.parents] if (p / "quant_fund_agent").is_dir()), None)
assert ROOT is not None, "run this from inside the QuantFundAgent repo"
os.chdir(ROOT); sys.path.insert(0, str(ROOT))

# --- load the OpenAI key from the repo's .env by EXPLICIT path ---
# (load_dotenv() with no arg resolves relative to *this file's* dir, which for a
#  notebook is notebooks/ — so we point it straight at the repo-root .env.)
from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

# --- run in-process (identical maths to the MCP server) + free embeddings ---
os.environ["QF_USE_MCP"]     = "0"      # evaluate candidates in-process
os.environ["QF_EMBEDDER"]    = "hash"   # no embedding API needed (RAG is off here)
os.environ["QF_FUNDAMENTALS"] = "0"     # OHLCV-only → fast, fully offline data

# --- surface the evolution loop's own logging inside the notebook ---
logging.basicConfig(level=logging.INFO, format="%(name)-18s %(message)s", stream=sys.stdout)
for noisy in ("httpx", "openai", "urllib3", "httpcore"):
    logging.getLogger(noisy).setLevel(logging.WARNING)

from quant_fund_agent.llm import resolve_research_model
MODEL = resolve_research_model()
has_key = bool(os.getenv("OPENAI_API_KEY"))

# cached tickers available offline
cache = ROOT / "data/market/yfinance/equity/1d"
n_cached = len(list(cache.glob("*.parquet"))) if cache.exists() else 0

print("repo root      :", ROOT)
print("OPENAI_API_KEY :", "present ✅" if has_key else "MISSING ❌ (LLM cells will fail)")
print("research model :", MODEL)
print("cached tickers :", n_cached, "(offline yfinance daily bars)")


The knobs below keep this run **small on purpose**.  The exact same objects, scaled
up, are what `run_factor_evolution.py` drives for a thesis run.


In [ ]:
N_TICKERS   = 8     # universe slice for the live demo scoring/evolution cells
FIELDS      = ["open", "high", "low", "close", "volume"]
HORIZON     = 6     # forecast/IC horizon in bars (daily → ~6 trading days)
DATA_DIR    = "ticker_data"   # ignored for yfinance (cache is keyed by config), kept for the API
SEED        = 7

WORKSPACE   = ROOT / "data/workspaces/yfinance_equity_sp100"
PREBOOK_PATH = WORKSPACE / "prebooks/lasso_prebook.json"
PREBOOK_MAX_FACTORS = 12       # live-demo cap; set to None in the builder to keep every non-zero Lasso name
PREBOOK_MAX_ROWS = 50_000

# a scratch dir for this run's checkpoints, OUTSIDE the repo (nothing is written
# into data/workspaces/ by the evolution loop itself, and nothing clutters git)
import tempfile
SCRATCH = Path(tempfile.gettempdir()) / "qfa_live_demo_evolution"
SCRATCH.mkdir(parents=True, exist_ok=True)
print("scratch out_dir:", SCRATCH)
print("prebook path   :", PREBOOK_PATH)


## 1 · The real data panel

Everything downstream reads this one dict of `field -> DataFrame(index=dates,
columns=tickers)`.  It is loaded through the project's data layer, which resolves
the active `quant.config.yaml` (a `yfinance` S&P-100 config) and serves the bars
straight from the parquet cache.


In [ ]:
from quant_fund_agent.data import load_panel

panel = load_panel(DATA_DIR, fields=FIELDS, n_tickers=N_TICKERS)
close = panel["close"]
print("fields loaded :", sorted(panel))
print("close panel   :", close.shape, "= (bars, tickers)")
print("tickers       :", list(close.columns))
print("date range    :", close.index.min().date(), "→", close.index.max().date())
close.tail(3)

In [ ]:
# a quick look at the raw material: normalised price paths for the slice
import matplotlib.pyplot as plt
%matplotlib inline

fig, ax = plt.subplots(figsize=(9, 3.2))
(close / close.iloc[0]).plot(ax=ax, lw=1)
ax.set_title("Real S&P-100 slice — prices normalised to 1.0 at the start")
ax.set_ylabel("growth of $1"); ax.legend(ncol=4, fontsize=7, loc="upper left")
plt.tight_layout(); plt.show()

## 2 · The data contract the LLM is given

Before any brainstorming, the researcher builds a **data context** — the exact
menu of fields the model is allowed to touch (anything off-menu makes a factor
*out-of-scope* and it is dropped).  It also carries a bar-size-aware **horizon
contract** so the model picks a sensible `prediction_horizon`.  This is the same
`build_data_context` the production pipeline uses.


In [ ]:
from quant_fund_agent.agents.factor_research.prompts import build_data_context

# infer seconds-per-bar straight from the real panel index (daily ≈ 86400 s)
_dt = close.index.to_series().diff().dropna().dt.total_seconds()
seconds_per_bar = float(_dt.median()) if len(_dt) else None

DATA_CONTEXT = build_data_context(sorted(FIELDS), seconds_per_bar)
print(f"(inferred seconds/bar = {seconds_per_bar:,.0f}  →  ~{seconds_per_bar/86400:.0f} day bars)\n")
print(DATA_CONTEXT)

## 3 · The pre-settled Lasso book

The residual and marginal-value axes only become meaningful once the candidate is
conditioned on a non-empty book.  For this live run the book is **fixed context**:
it is not inserted into the evolutionary archive and it will not be persisted as a
new discovery.  It simply gives the evaluator something real to residualise
against.

The JSON artifact below was built from all factor records in
`data/workspaces/yfinance_equity_sp100` by fitting a Lasso model to predict the
horizon-6 forward return.  The live notebook keeps the top 12 Lasso coefficients
so scoring stays quick; the builder can be rerun with a different cap for larger
experiments.


In [ ]:
from quant_fund_agent.research_eval.prebook import (
    book_entries,
    fit_lasso_prebook,
    load_prebook,
    load_workspace_programs,
    save_prebook,
)

if PREBOOK_PATH.exists():
    prebook = load_prebook(PREBOOK_PATH)
else:
    # Slow path: only runs if the reusable artifact is missing.
    programs, load_dropped = load_workspace_programs(WORKSPACE)
    prebook = fit_lasso_prebook(
        programs, panel,
        target_horizon=HORIZON,
        max_rows=PREBOOK_MAX_ROWS,
        max_members=PREBOOK_MAX_FACTORS,
        seed=SEED,
    )
    prebook["workspace"] = str(WORKSPACE)
    prebook["load_dropped"] = load_dropped
    prebook["fields"] = FIELDS
    prebook["n_tickers"] = N_TICKERS
    save_prebook(prebook, PREBOOK_PATH)

assert prebook.get("ok"), prebook.get("error", "prebook build failed")
PRESETTLED_BOOK = book_entries(prebook)
assert PRESETTLED_BOOK, "Lasso prebook is empty; widen/relax the Lasso selection before scoring residual effects"

prebook_df = pd.DataFrame([
    {
        "rank": f.get("rank"),
        "factor_id": f["factor_id"],
        "coef": f.get("coefficient"),
        "importance": f.get("importance"),
        "source": f.get("source_prerun"),
    }
    for f in prebook.get("factors", [])
])

print(f"pre-settled book : {len(PRESETTLED_BOOK)} factor(s)")
print(f"source factors   : {prebook.get('n_usable')}/{prebook.get('n_candidates')} usable; "
      f"{prebook.get('n_lasso_nonzero')} non-zero Lasso coefficients")
print(f"lasso alpha      : {prebook.get('alpha')} ({prebook.get('alpha_source')})")
print(f"artifact         : {PREBOOK_PATH}")
prebook_df


## 4 · Seeding, step 1 — **brainstorm** (live LLM)

Generation 0 comes from the *same* brainstorm/codegen path the one-shot baseline
uses, so both arms start from the same generator.  Here we call the real model to
propose a few factor **ideas** — each a structured hypothesis: a mechanism, an
`expected_sign`, and a chosen `prediction_horizon`.  Nothing is scored yet.


In [ ]:
from quant_fund_agent.agents.factor_research.graph import _brainstorm_one
from quant_fund_agent.llm import make_chat_llm

brainstorm_llm = make_chat_llm(temperature=0.7, timeout=120, max_retries=3)

t0 = time.time()
raw_ideas = _brainstorm_one(brainstorm_llm, paper=None, n_ideas=3,
                            known_ids=set(), session_id="live-demo",
                            data_context=DATA_CONTEXT)
print(f"brainstorm returned {len(raw_ideas)} idea(s) in {time.time()-t0:.1f}s\n")

for i, idea in enumerate(raw_ideas):
    print(f"── idea {i} ─────────────────────────────────────────────")
    print(f"  factor_id  : {idea.get('factor_id')}")
    print(f"  name       : {idea.get('name')}")
    print(f"  category   : {idea.get('category')}")
    print(f"  horizon    : {idea.get('prediction_horizon')}  "
          f"(suggested {idea.get('suggested_horizons')})")
    print(f"  exp. sign  : {idea.get('expected_sign')}")
    print("  idea       :", textwrap.fill(idea.get('trading_idea',''), 92,
                                          subsequent_indent=' '*15))
    print()

## 5 · Seeding, step 2 — **codegen** (live LLM)

Each surviving idea is handed to a lower-temperature model that writes an actual
`BaseFactor` subclass.  The code is compiled **in-memory** and smoke-tested (a
candidate only becomes a file if it survives evolution).  If compilation fails, the
error is fed back for **one self-correcting retry** — the same loop the production
codegen uses.


In [ ]:
from quant_fund_agent.agents.factor_research.evolution.loop import _codegen_program, coerce_idea

codegen_llm = make_chat_llm(temperature=0.2, timeout=120, max_retries=3)

seed_prog = {}
for i, raw in enumerate(raw_ideas):
    idea = coerce_idea(raw, HORIZON)
    prog = _codegen_program(codegen_llm, idea, DATA_CONTEXT)   # compiles in-memory
    if prog is None:
        raise RuntimeError(f"codegen failed for idea {i}: {idea.factor_id}")
    seed_prog[i] = prog

    print("compiled factor :", prog.factor_id, "| horizon", prog.prediction_horizon,
          "| expected_sign", prog.expected_sign)
    print("=" * 92)
    print(prog.code)


## 6 · The reward channel — **real ML fitting** (no LLM)

Now the deterministic part.  `evaluate_fitness` compiles the candidate, computes
its signal on the panel, and scores it on a **5-axis Pareto vector** against the
pre-settled Lasso book.  The primary axis is **marginal value (LOCO)**: a
**gradient-boosting** model is fit on the in-sample window to combine the fixed
book's signals into one forward-return forecast, and we measure how much the
candidate *adds* to the combined model's **out-of-sample IC**.

Because the book is already non-empty, the second and third seed ideas now show
real residual/orthogonal effects immediately instead of being evaluated as if
they were first in line.  Around marginal value sit four more axes —

- **independence** = the candidate's **residual (orthogonalised) IC** (its edge in
  the direction the book doesn't already span),
- **robustness** (CPCV mean − λ·std, − plateau, + sign bonus),
- **parsimony** (−AST complexity), and
- **regime_independence** = its marginal ΔIC **on the crash/stress bars** — edge
  added exactly where the book is weakest —

behind hard **gates** (coverage / degradation / deflation) a factor must clear.


In [ ]:
from quant_fund_agent.mcp import research_client


def _ax(v, nd=4):
    return f"{v:+.{nd}f}" if v is not None else "n/a"


t0 = time.time()
fit = {}
for i, prog in seed_prog.items():
    res = research_client.evaluate_fitness(
        candidate={"factor_id": prog.factor_id, "code": prog.code,
                   "expected_sign": prog.expected_sign},
        book=PRESETTLED_BOOK,          # fixed Lasso book → residual/marginal effects are visible immediately
        jitter=[],                     # (the plateau probe rides along here in a real run)
        target_horizon=HORIZON, is_frac=0.6, val_frac=0.2, n_trials=i + 1,
        cpcv_groups=4, cpcv_k=2, embargo=0,
        data_dir=DATA_DIR, n_tickers=N_TICKERS, fields=FIELDS,
        # nonlinear LOCO combiner (captures conditioning value), residual-IC
        # independence, drawdown regime axis
        marginal_model="gradient_boosting",
        independence_metric="residual_ic", regime_kind="drawdown", regime_quantile=0.2,
    )
    print(f"Factor {prog.factor_id} scored in {time.time()-t0:.1f}s  |  ok = {res['ok']}\n")
    if not res.get("ok"):
        print(res.get("error"))
        print("=" * 92)
        continue

    fit[i] = res["fitness"]
    obj = fit[i]["objective"]
    d = fit[i]["diagnostics"]

    print("── Pareto objective vector (5 axes, all maximised) ─")
    print(f"  base_book_ic  (fixed book OOS IC)        : {_ax(d.get('base_ic'))}")
    print(f"  with_factor_ic(book + candidate OOS IC) : {_ax(d.get('with_ic'))}")
    print(f"  marginal_value (LOCO ΔOOS-IC, primary) : {_ax(obj['marginal_value'])}")
    print(f"  independence   (residual/orthogonal IC): {_ax(obj['independence'])}")
    print(f"  robustness     (CPCV mean−λ·std, …)    : {_ax(obj['robustness'])}")
    print(f"  parsimony      (−AST complexity)       : {_ax(obj['parsimony'], 1)}")
    print(f"  regime_indep.  (ΔIC on crash bars)     : {_ax(obj['regime_independence'])}"
          f"   (stress bars scored: {d.get('n_stress_obs')})")
    print("\n── hard gates ──────────────────────────────────────")
    g = fit[i]["gates"]
    print(f"  passed = {g['passed']}   "
          f"(coverage={g['coverage_ok']}, degradation={g['degradation_ok']}, "
          f"deflation={g['deflation_ok']})")
    if g.get("reasons"):
        for k, v in g["reasons"].items():
            print(f"    ✗ {k}: {v}")
    print("=" * 92)


It is completely normal for a first random idea to **fail a gate** (a degradation
gate fires when the edge does not survive out-of-sample).  A failing candidate is
not thrown away silently — its diagnostics become the *teacher signal* for the next
step.


## 7 · Reflection — the deterministic **teacher** (no LLM writes it)

The numeric dashboard is turned into a natural-language **mutation brief** by
rule-based logic — *no LLM writes this*.  Every advice line is triggered by a
threshold on the numbers above.  This brief is the only feedback the mutating LLM
receives about how its parent performed.


In [ ]:
from quant_fund_agent.research_eval.fitness import FitnessResult
from quant_fund_agent.agents.factor_research.evolution.reflection import mutation_brief

for i, prog in seed_prog.items():
    if i not in fit:
        continue
    print(f"── reflection for {prog.factor_id} ───────────────────────")
    fitness = FitnessResult.from_dict(fit[i])
    brief = mutation_brief(fitness, book_size=len(PRESETTLED_BOOK))
    print(brief)
    print("=" * 92)


## 8 · Mutation — the creative operator (live LLM)

The parent program **plus its reflection brief** are handed back to the LLM, which
proposes a *child* — a genuinely new factor that tries to fix what the brief
flagged.  The child is parsed, its id de-collided, and compiled in-memory (with the
same one-shot feedback retry).  Below we show the parent → child jump.


In [ ]:
from quant_fund_agent.agents.factor_research.evolution.mutation import (
    build_mutation_prompt, parse_child_response,
)
from quant_fund_agent.factors.inmem import compile_factor

# Mutate the seed idea with the best marginal value against the fixed book.
parent_i = max(
    fit,
    key=lambda j: fit[j]["objective"].get("marginal_value")
    if fit[j]["objective"].get("marginal_value") is not None else float("-inf"),
)
parent = seed_prog[parent_i]
parent_fitness = FitnessResult.from_dict(fit[parent_i])
parent_brief = mutation_brief(parent_fitness, book_size=len(PRESETTLED_BOOK))
existing_ids = [p.factor_id for p in seed_prog.values()] + [b["factor_id"] for b in PRESETTLED_BOOK]

prompt = build_mutation_prompt(parent, parent_brief, DATA_CONTEXT, existing_ids=existing_ids)

# what the model actually receives (trimmed) — note the brief is embedded verbatim
print("MUTATION PROMPT (excerpt) ".ljust(92, "─"))
print("…parent idea + code + the reflection brief + a strict JSON schema…\n")
print("brief handed to the model:\n" + textwrap.indent(parent_brief.split(chr(10))[0], "    "), "…\n")

resp = brainstorm_llm.invoke(prompt)
child = parse_child_response(getattr(resp, "content", str(resp)))
compile_factor(child.code, child.factor_id, smoke=True)   # validate the child

print("PARENT →", parent.factor_id)
print("  idea :", textwrap.fill(parent.trading_idea or parent.description, 88,
                                subsequent_indent=' '*9))
print("\nCHILD  →", child.factor_id, f"(horizon {child.prediction_horizon}, sign {child.expected_sign})")
print("  idea :", textwrap.fill(child.trading_idea or child.description, 88,
                                subsequent_indent=' '*9))
print("\nchild code:\n" + "=" * 92)
print(child.code)


## 9 · The whole loop, turning on its own

Sections 3–7 walked one candidate through one lap by hand.  `EvolutionLoop` does
exactly that in a cycle: **seed → (select parent by NSGA-II tournament → mutate /
crossover / jitter → score → insert → reflect)\***, keeping a gate-passing **Pareto
archive** that *is* the accepted book, with per-generation checkpoints.

We run a deliberately tiny configuration (2 generations, a handful of children).
Watch the log: you will see the real brainstorm, the codegen, each candidate being
scored, and the archive / `n_trials` growing.  `n_trials` is the deflation counter —
every *scored* look is billed so the statistics stay honest.


In [ ]:
from quant_fund_agent.agents.factor_research.evolution.loop import (
    EvolutionLoop, EvolutionRunConfig,
)

cfg = EvolutionRunConfig(
    generations=2, population_size=6, children_per_generation=2,
    n_seed_ideas=3, seed_papers=0, retrieval="none",
    p_llm_semantic=0.6, p_crossover=0.2, p_jitter=0.2,
    target_horizon=HORIZON, is_frac=0.6, val_frac=0.2,
    cpcv_groups=4, cpcv_k=2, embargo=0,
    data_dir=DATA_DIR, n_tickers=N_TICKERS, seed=SEED,
    fixed_book=PRESETTLED_BOOK,
    out_dir=str(SCRATCH),
)

# pass the real fields + our correct daily data-context so the loop is consistent
loop = EvolutionLoop(cfg, data_context=DATA_CONTEXT, fields=FIELDS)

print(f"fixed conditioning book: {len(loop.fixed_book)} factor(s); archive starts empty")
t0 = time.time()
summary = loop.run()          # ← real LLM + real ML, generation by generation
print(f"\n{'='*70}\nfinished in {time.time()-t0:.0f}s")
print(json.dumps({k: summary[k] for k in
                  ('generations','n_trials','population','fixed_book_size','n_eval_failures','elapsed_sec')},
                 indent=2))


## 10 · Results — the Pareto archive (the accepted book)

The table below is the run's newly accepted archive.  It deliberately excludes the
pre-settled Lasso book: those factors were fixed conditioning context, not new
survivors discovered by this mini-run.


In [ ]:
import pandas as pd, numpy as np

def _r(x, nd):                          # round, but tolerate a None axis
    return round(x, nd) if x is not None else np.nan

rows = []
for eg in loop.controller.archive:
    o = eg.fitness.objective
    rows.append({
        "factor_id": eg.genome.factor_ids[0],
        "gen": eg.genome.generation,
        "operator": eg.genome.operator,
        "marginal_value": _r(o.marginal_value, 4),
        "independence": _r(o.independence, 4),       # residual IC
        "robustness": _r(o.robustness, 4),
        "parsimony": _r(o.parsimony, 1),
        "regime_indep": _r(o.regime_independence, 4),  # crash-complementarity
        "gate_pass": eg.fitness.gates.passed,
    })
archive_df = pd.DataFrame(rows)
if not archive_df.empty:
    archive_df = archive_df.sort_values("marginal_value", ascending=False).reset_index(drop=True)

print(f"archive (Pareto front): {len(archive_df)}   |   "
      f"kept_pool (every gate-passer): {len(loop.controller.kept_pool)}   |   "
      f"n_trials billed: {loop.controller.n_trials}")
print("(with --curation greedy|elastic_net the persisted book is curated from the "
      "kept_pool, not limited to the archive front)")
archive_df

In [ ]:
# the two axes the search trades off: marginal OOS edge vs robustness
fig, ax = plt.subplots(figsize=(6.5, 4.2))
for _, r in archive_df.dropna(subset=["robustness", "marginal_value"]).iterrows():
    ax.scatter(r.robustness, r.marginal_value,
               s=90, c=("tab:green" if r.gate_pass else "tab:red"),
               edgecolor="k", zorder=3)
    ax.annotate(r.factor_id[:14], (r.robustness, r.marginal_value),
                fontsize=7, xytext=(4, 4), textcoords="offset points")
ax.axhline(0, color="grey", lw=.7); ax.axvline(0, color="grey", lw=.7)
ax.set_xlabel("robustness  (CPCV mean − λ·std, sign-consistency, − plateau penalty)")
ax.set_ylabel("marginal value  (LOCO ΔOOS-IC)")
ax.set_title("Pareto archive — green = clears every hard gate")
plt.tight_layout(); plt.show()

In [ ]:
# lineage: who each survivor descends from (the last few admitted genomes)
print("lineage tail (genome ← operator ← parents):")
for row in loop.controller.lineage[-6:]:
    print(f"  gen{row.get('generation')}  {row.get('operator'):11}  "
          f"{row.get('genome_id')}  ← {row.get('parent_ids') or '(seed)'}")

## 11 · From here to a thesis run

That was the whole machine in miniature, running for real.  Nothing here was
persisted — `persist_archive(...)` is what materialises the final book into
`data/workspaces/<config>/preruns/<name>/`, after which the comparison and
walk-forward harnesses score it exactly like a one-shot prerun.

To scale the *same* objects up (papers, islands, a stronger model, RAG/GraphRAG
grounding, the Hypothesis→Debate→Codegen split, and **two-stage curation**):

```bash
# a real evolution prerun (persisted); keep every gate-passer, curate a 12-factor book
python run_factor_evolution.py --name evo1 --generations 12 --population 12 \
    --children-per-gen 10 --islands 2 --seed-papers 6 --retrieval graphrag \
    --debate on --model gpt-4o --curation greedy --n-keep 12 \
    --fixed-book data/workspaces/yfinance_equity_sp100/prebooks/lasso_prebook.json

# then walk-forward validation of the whole loop
python run_factor_evolution.py --name evo1 \
    --fixed-book data/workspaces/yfinance_equity_sp100/prebooks/lasso_prebook.json \
    --walk-forward 2023-01-01,2024-01-01,2025-01-01
```

`--fixed-book` conditions candidate fitness on the Lasso book without persisting those factors as discoveries.

`--curation archive` (the default) persists the Pareto front (the one-stage
behaviour); `--curation greedy|elastic_net` persists the curated `kept_pool`
instead, so a good factor is not discarded merely for being dominated.  The
independence axis defaults to residual IC (`--independence-metric
delta_participation` restores the legacy Δ-participation-ratio for an ablation),
and the regime axis is configurable via `--regime-kind {drawdown,volatility}`.

For the conceptual deep-dive on each component (splits, deflation, the 5-axis
objective vector, the NSGA-II controller, two-stage curation, RAG/GraphRAG) see the
companion notebook `evolutionary_factor_researcher_walkthrough.ipynb`.
